In [ ]:
from dataclasses import replace
from pathlib import Path

import functions
from functions import (
    ConfigTrainClassifier,
    DatasetSpec,
    PolicySpec,
    RewardSpec,
    TrainingPPOConfig,
)

In [ ]:
policy_spec = PolicySpec(
    model_name="Qwen/Qwen3-0.6B",
)

proxy_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-0.6B",
    mode_name="proxy",
)

judge_spec = RewardSpec(
    class_name="RewardModel",
    model_name="Skywork/Skywork-Reward-V2-Qwen3-4B",
    mode_name="judge",
)

classifier_config = ConfigTrainClassifier(
    policy=policy_spec,
    reward=proxy_spec,
    judge=judge_spec,
    start_dataset=0,
    end_dataset=200,
)

gap_calibration = functions.calculate_gap_calibration(
    classifier_config
)
gap_calibration

In [ ]:
normalized_proxy_spec = replace(
    proxy_spec,
    mean=gap_calibration.proxy_mean,
    std=gap_calibration.proxy_std,
)

dataset_spec = DatasetSpec(end=200)

first_ppo_config = TrainingPPOConfig(
    policy=policy_spec,
    reward=normalized_proxy_spec,
    dataset=dataset_spec,
    output_dir="outputs/ppo_without_classifier",
)

first_policy = functions.temp_ppo_train_policy(first_ppo_config)

first_policy_path = Path(first_ppo_config.output_dir) / "final"

In [ ]:
new_classifier, reused_calibration = functions.create_classifier(
    classifier_config,
    policy=first_policy,
    calibration=gap_calibration,
)

assert reused_calibration is gap_calibration
new_classifier.checkpoint_path

In [ ]:
second_policy_spec = replace(
    policy_spec,
    checkpoint=first_policy_path,
)

second_ppo_config = replace(
    first_ppo_config,
    policy=second_policy_spec,
    output_dir="outputs/ppo_with_classifier",
)

second_policy = functions.temp_ppo_train_policy(
    second_ppo_config,
    classifiers=(new_classifier,),  # use the in-memory classifier
)

second_policy_path = Path(second_ppo_config.output_dir) / "final"
second_policy_path